**[Source]** New (통합 프로젝트) + Donghwan Project(최종 평가 지표) + Jisoo Project(생성·ROUGE 평가 정의)
**[Status]** ADAPTED
**[Role]** 잠긴 최종 설정으로 Test를 한 번만 생성·평가
**[Modification]** Test 접근은 13번의 FINAL_LOCKED와 설정 해시가 일치할 때만 가능. 이미 한 번 실행했으면 재실행을 막는다.
**이 노트북은 GPU가 필요하며 작성 환경에서 실행하지 못했다(RUN_TEST=False). 실행 후 결과를 채워야 한다.**

# 14. 최종 Test 평가 (1회)
- 전제: 13번에서 설정이 잠겨 있어야 한다(`config/FINAL_LOCKED`, `final_model_config.json` 해시 일치).
- **Test 결과를 보고 모델·디코딩·후처리·규칙을 바꾸지 않는다.** 결과가 마음에 들지 않아도 그대로 보고한다.
- 보고 항목: 입력 복사 기준선 대비, NFKC/raw 지표, 후처리 전후, 문서 단위 CI, Train에 없는 입력 기준, 오류 유형 분포. **Validation 수치와 Test 수치는 다른 데이터에서 계산된 것이므로 ‘개선’으로 해석하지 않는다.**
- `RUN_TEST=False`가 기본이다. GPU와 체크포인트가 준비된 환경에서 `True`로 바꿔 딱 한 번 실행한다.

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 잠금 확인 (Test를 열기 전 반드시 통과해야 함)
RUN_TEST = False
LOCK = json.loads((P.CONFIG / "FINAL_LOCKED").read_text(encoding="utf-8")); CFG = json.loads((P.CONFIG / "final_model_config.json").read_text(encoding="utf-8"))
assert common.sha256_file(P.CONFIG / "final_model_config.json") == LOCK["final_model_config_sha256"], "최종 설정이 잠금 이후 바뀌었습니다 → Test 평가 중단"
DONE = P.RUNS / "test_eval_done.flag"; assert not DONE.exists(), "Test 평가는 이미 1회 수행되었습니다. 재실행 금지(결과를 본 뒤 재평가하면 튜닝이 됨)"
print("잠금 확인 통과 | 모델:", CFG["model"]["name"], "| 디코딩:", CFG["decoding"]["strategy"], "| 후처리:", CFG["postprocess"]["apply"], "| RUN_TEST =", RUN_TEST)

잠금 확인 통과 | 모델: paust/pko-t5-base | 디코딩: greedy | 후처리: True | RUN_TEST = False


In [3]:
# [셀 2] Test 생성·평가 — RUN_TEST=True이고 GPU가 있을 때만 (작성 환경에서는 실행하지 못함: 실행 후 확인)
try:
    import torch; HAS_GPU = torch.cuda.is_available()
except Exception:
    HAS_GPU = False
CKDIR = P.J_OUT / "checkpoints" / "pkot5_full_1epoch"
if not (RUN_TEST and HAS_GPU and CKDIR.exists()):
    print("Test 평가 미실행: RUN_TEST=%s, GPU=%s, checkpoint=%s" % (RUN_TEST, HAS_GPU, CKDIR.exists())); print("→ Test 성능은 아직 없다. 어떤 Test 수치도 주장할 수 없다.")
else:
    import seq2seq_tools as S2S, ko_postprocess
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    assert common.sha256_file(CKDIR / "model.safetensors") == CFG["model"]["checkpoint_weight_sha256"], "체크포인트 해시 불일치"
    TEST = common.read_split(P, "test", columns=["document_id", "utterance_id", "input", "target", "flag_seen_input_in_train"], allow_test=True)
    SRC, TGT, DOC = [r["input"] for r in TEST], [r["target"] for r in TEST], [r["document_id"] for r in TEST]
    tok = AutoTokenizer.from_pretrained(str(CKDIR)); model = AutoModelForSeq2SeqLM.from_pretrained(str(CKDIR)).cuda().eval()
    d = CFG["decoding"]; gk = {"num_beams": d["num_beams"], "do_sample": d["do_sample"]}
    PRED_RAW = S2S.generate_resumable(model, tok, SRC, P.RUNS / "test_pred_part.jsonl", CFG["tokenizer"]["max_length_input"], d["max_new_tokens"], gk, batch_size=64)
    PRED = [ko_postprocess.restore_compat_jamo(p) for p in PRED_RAW] if CFG["postprocess"]["apply"] else PRED_RAW
    need = np.array([s != t for s, t in zip(SRC, TGT)]); docs = np.array(DOC)
    OUT = {"n": len(TEST), "docs": len(set(DOC)), "final(후처리 적용)": {nf: km.generation_metrics(SRC, TGT, PRED, nfkc=(nf == "NFKC")) for nf in ("NFKC", "raw")}, "model_only": {nf: km.generation_metrics(SRC, TGT, PRED_RAW, nfkc=(nf == "NFKC")) for nf in ("NFKC", "raw")},
           "input_copy": {nf: km.generation_metrics(SRC, TGT, SRC, nfkc=(nf == "NFKC")) for nf in ("NFKC", "raw")}}
    R2 = km.score_rows(PRED, TGT); R2c = km.score_rows(SRC, TGT); ex = np.array([km.normalize_for_evaluation(p) == km.normalize_for_evaluation(t) for p, t in zip(PRED, TGT)])
    OUT["ci95"] = {"어절 R2": km.boot_mean(R2["R2_어절"], docs), "Balanced EM(NFKC)": km.boot_balanced_em(ex, need, docs), "R2 차이(모델−복사)": km.boot_paired_diff(R2["R2_어절"], R2c["R2_어절"], docs)}
    OUT["rouge"] = {k: float(v.mean()) for k, v in R2.items()}
    seen = np.array([bool(r["flag_seen_input_in_train"]) for r in TEST]); idx = np.where(~seen)[0]
    OUT["unseen_input_only"] = km.generation_metrics([SRC[i] for i in idx], [TGT[i] for i in idx], [PRED[i] for i in idx], with_chrf=False)
    (P.RUNS / "test_eval_14.json").write_text(json.dumps(OUT, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
    pd.DataFrame({"utterance_id": [r["utterance_id"] for r in TEST], "input": SRC, "target": TGT, "prediction_raw": PRED_RAW, "prediction_final": PRED}).to_csv(P.RUNS / "test_predictions_14.csv", index=False, encoding="utf-8-sig")
    DONE.write_text(time.strftime("%Y-%m-%d %H:%M:%S")); print(json.dumps({k: v for k, v in OUT.items() if k in ("n", "docs", "ci95", "rouge")}, ensure_ascii=False, indent=1, default=float))

Test 평가 미실행: RUN_TEST=False, GPU=False, checkpoint=True
→ Test 성능은 아직 없다. 어떤 Test 수치도 주장할 수 없다.


## 해석
이번 작성·실행에서는 **Test를 평가하지 않았다**(GPU 없음, RUN_TEST=False). 따라서 이 프로젝트의 Test 성능은 **없다**. 위 셀을 GPU 환경에서 한 번 실행한 뒤 결과를 이 셀에 기록해야 한다. 실행 결과가 Validation과 다르더라도 설정을 바꾸지 않는다.